In [ ]:
!pip install -q arxiv langchain langchain-community langchain-huggingface langchain-chroma chromadb pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [ ]:
import os
import time
import arxiv
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from google.colab import userdata
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')

/tmp/ipykernel_2847/751504025.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
PDF_DIR = "./papers"
CHROMA_DIR = "./chroma_db"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

In [ ]:
PAPER_IDS = [
    "1706.03762",  # Attention Is All You Need
    "1810.04805",  # BERT
    "2005.11401",  # Retrieval-Augmented Generation (Lewis et al.)
    "1910.13461",  # BART
    "2004.04906",  # Dense Passage Retrieval (DPR)
    "1908.10084",  # Sentence-BERT - the architecture behind all-MiniLM-L6-v2
    "2104.08821",  # SimCSE - contrastive sentence embeddings
    "2004.12832",  # ColBERT - late-interaction dense retrieval
    "2007.01282",  # Fusion-in-Decoder - retrieval-augmented generation, FAISS-based
    "1603.09320",  # HNSW - the ANN algorithm most vector DBs (incl. Chroma) use under the hood
    "1910.10683",  # T5 - text-to-text transfer transformer, useful generation baseline
]

In [ ]:
os.makedirs(PDF_DIR, exist_ok=True)

In [ ]:
def fetch_papers(arxiv_ids):
    """Download PDFs and collect metadata for each paper."""
    client = arxiv.Client()
    search = arxiv.Search(id_list=arxiv_ids)
    papers = []
    for result in client.results(search):
        arxiv_id = result.get_short_id().split("v")[0]  # strip version suffix
        filename = f"{arxiv_id}.pdf"
        filepath = os.path.join(PDF_DIR, filename)
        if not os.path.exists(filepath):
            pdf_url = next((l.href for l in result.links if l.title == "pdf"), None)
            if pdf_url is None:
                pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"  # fallback
            response = requests.get(pdf_url, timeout=30)
            response.raise_for_status()
            with open(filepath, "wb") as f:
                f.write(response.content)
            time.sleep(1)  # be polite to arXiv's servers
        papers.append({
            "arxiv_id": arxiv_id,
            "title": result.title,
            "authors": [a.name for a in result.authors[:3]],
            "filepath": filepath,
        })
        print(f"Fetched: {result.title}")
    return papers

In [ ]:
def load_and_chunk(papers):
    """Load each PDF page by page and split into chunks, tagging each with metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    all_chunks = []
    for paper in papers:
        loader = PyPDFLoader(paper["filepath"])
        pages = loader.load()
        chunks = splitter.split_documents(pages)
        for chunk in chunks:
            chunk.metadata.update({
                "arxiv_id": paper["arxiv_id"],
                "title": paper["title"],
                "authors": paper["authors"],          # now a list
                "page": chunk.metadata.get("page", 0) + 1,
                "source": f"{paper['title'][:50]} (arXiv:{paper['arxiv_id']}, p.{chunk.metadata.get('page', 0) + 1})",
            })
        all_chunks.extend(chunks)
        print(f"  {paper['title'][:50]}... -> {len(chunks)} chunks")
    return all_chunks


In [ ]:
def build_vectorstore(chunks, reset=False):
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    if reset or not os.path.exists(CHROMA_DIR):
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embeddings,
            persist_directory=CHROMA_DIR,
            collection_name="arxiv_papers",
        )
    else:
        vectorstore = Chroma(
            persist_directory=CHROMA_DIR,
            embedding_function=embeddings,
            collection_name="arxiv_papers",
        )
        vectorstore.add_documents(chunks)
    return vectorstore

In [ ]:
def sanity_check(vectorstore):
    query = "What is self-attention?"
    results = vectorstore.similarity_search(query, k=3)
    print(f"\nTest query: '{query}'")
    for i, doc in enumerate(results, 1):
        print(f"\n  Result {i}")
        print(f"  Source: {doc.metadata.get('source', 'MISSING - check metadata')}")
        print(f"  Preview: {doc.page_content[:200]}...")

In [ ]:
import json

def save_papers_index(papers):
    index = [{"arxiv_id": p["arxiv_id"], "title": p["title"], "authors": p["authors"]} for p in papers]
    with open("./papers_index.json", "w") as f:
        json.dump(index, f, indent=2)
    print(f"Saved index with {len(index)} papers")

In [ ]:
if __name__ == "__main__":
    print("Step 1/4: Fetching papers from arXiv...")
    papers = fetch_papers(PAPER_IDS)

    print("\nStep 2/4: Loading and chunking PDFs...")
    chunks = load_and_chunk(papers)
    print(f"\nTotal chunks created: {len(chunks)}")

    print("\nStep 3/4: Embedding and storing in ChromaDB...")
    vectorstore = build_vectorstore(chunks)
    print(f"Stored at: {CHROMA_DIR}")

    print("\nStep 4/4: Sanity check...")
    sanity_check(vectorstore)

    save_papers_index(papers)

Step 1/4: Fetching papers from arXiv...
Fetched: Attention Is All You Need
Fetched: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
Fetched: ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT
Fetched: Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks
Fetched: Dense Passage Retrieval for Open-Domain Question Answering
Fetched: Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer
Fetched: BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension
Fetched: Leveraging Passage Retrieval with Generative Models for Open Domain Question Answering
Fetched: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
Fetched: SimCSE: Simple Contrastive Learning of Sentence Embeddings
Fetched: Efficient and robust approximate nearest neighbor search using Hierarchical Navigable Small World graphs

Step 2/4: Loading and chu

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored at: ./chroma_db

Step 4/4: Sanity check...

Test query: 'What is self-attention?'

  Result 1
  Source: Exploring the Limits of Transfer Learning with a U (arXiv:1910.10683, p.16)
  Preview: Raffel, Shazeer, Roberts, Lee, Narang, Matena, Zhou, Li and Liu
x1 x2 x3 x4 x5
y5
y4
y3
y2
y1
x1 x2 x3 x4 x5
y5
y4
y3
y2
y1
x1 x2 x3 x4 x5
y5
y4
y3
y2
y1
Figure 3: Matrices representing different atte...

  Result 2
  Source: Attention Is All You Need (arXiv:1706.03762, p.6)
  Preview: versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
dur...

  Result 3
  Source: Exploring the Limits of Transfer Learning with a U (arXiv:1910.10683, p.5)
  Preview: Exploring the Limits of Transfer Learning
mechanism after each self-attention layer that attends to the output of the encoder. The
self-attention mechanism in the decoder also uses a form of autoregre...

In [ ]:
!pip install -q fastapi uvicorn nest_asyncio langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00


In [ ]:
import os
import json
import threading
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

nest_asyncio.apply()

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="arxiv_papers",
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=0.3,
)

print(f"Vectorstore loaded: {vectorstore._collection.count()} chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore loaded: 944 chunks


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# step 1: rewrite question using chat history so follow-ups work
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and latest question, rewrite the question "
     "to be fully self-contained. Return only the rewritten question, nothing else."
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# step 2: answer using retrieved chunks
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a research paper assistant. Answer using only the context below "
     "from indexed arXiv papers. Always mention which paper your answer is from. "
     "If the context doesn't contain enough information, say so clearly.\n\n"
     "Context:\n{context}"
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def run_rag(question, chat_history=[]):
    # rewrite question if there's history
    if chat_history:
        rewrite_chain = contextualize_prompt | llm | StrOutputParser()
        standalone_question = rewrite_chain.invoke({
            "input": question,
            "chat_history": chat_history,
        })
    else:
        standalone_question = question

    docs = retriever.invoke(standalone_question)

    answer_chain = qa_prompt | llm | StrOutputParser()
    answer = answer_chain.invoke({
        "input": question,
        "chat_history": chat_history,
        "context": format_docs(docs),
    })

    return {"answer": answer, "context": docs}

print("RAG chain ready")

RAG chain ready


In [ ]:
app = FastAPI(title="Research Paper RAG API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    question: str
    chat_history: list = []  # [{"role": "user"/"assistant", "content": "..."}]

class IngestRequest(BaseModel):
    arxiv_id: str

# --- helpers ---

def to_langchain_history(history):
    """Convert Streamlit-style dicts to LangChain message objects."""
    messages = []
    for msg in history:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))
    return messages

def extract_sources(docs):
    """Deduplicate and format source citations from retrieved chunks."""
    seen = set()
    sources = []
    for doc in docs:
        arxiv_id = doc.metadata.get("arxiv_id")
        page = doc.metadata.get("page", "?")
        key = (arxiv_id, page)
        if key not in seen:
            seen.add(key)
            sources.append({
                "title": doc.metadata.get("title", ""),
                "arxiv_id": arxiv_id,
                "page": page,
                "source": doc.metadata.get("source", ""),
            })
    return sources

# --- endpoints ---

@app.get("/health")
def health():
    return {"status": "ok", "chunks": vectorstore._collection.count()}

@app.get("/papers")
def list_papers():
    import os
    print("Current dir:", os.getcwd())
    print("Files here:", os.listdir("."))
    try:
        with open("./papers_index.json") as f:
            return json.load(f)
    except FileNotFoundError:
        raise HTTPException(status_code=404, detail="papers_index.json not found — run ingestion first")
@app.post("/chat")
def chat(req: ChatRequest):
    try:
        history = to_langchain_history(req.chat_history)
        result = run_rag(req.question, history)
        return {
            "answer": result["answer"],
            "sources": extract_sources(result["context"]),
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/ingest")
def ingest_paper(req: IngestRequest):
    try:
        arxiv_id = req.arxiv_id.strip()
        new_papers = fetch_papers([arxiv_id])          # reuses Phase 1 function
        new_chunks = load_and_chunk(new_papers)        # reuses Phase 1 function
        build_vectorstore(new_chunks, reset=False)     # incremental add

        # update papers_index.json
        try:
            with open("./papers_index.json") as f:
                index = json.load(f)
        except FileNotFoundError:
            index = []
        existing_ids = {p["arxiv_id"] for p in index}
        for p in new_papers:
            if p["arxiv_id"] not in existing_ids:
                index.append({"arxiv_id": p["arxiv_id"], "title": p["title"], "authors": p["authors"]})
        with open("./papers_index.json", "w") as f:
            json.dump(index, f, indent=2)

        return {"message": f"Ingested {len(new_chunks)} chunks from {new_papers[0]['title']}"}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import subprocess, time, threading

# start FastAPI in background
thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True,
)
thread.start()
time.sleep(2)

# download and start cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nohup ./cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1 &

time.sleep(8)

# print the public URL
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | head -1

INFO:     Started server process [2847]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


cloudflared: Text file busy
https://response-adsl-remind-partly.trycloudflare.com


In [ ]:
import requests as req

# read URL from tunnel log
with open("tunnel.log") as f:
    content = f.read()

import re
match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', content)
BASE = match.group(0) if match else None
print("Backend URL:", BASE)

# test /health
print(req.get(f"{BASE}/health", timeout=10).json())

# test /papers
print(req.get(f"{BASE}/papers", timeout=10).json())

# test /chat
r1 = req.post(f"{BASE}/chat", json={"question": "What is self-attention?", "chat_history": []}, timeout=30)
if r1.status_code == 200:
    response_json = r1.json()
    print("\nAnswer:", response_json["answer"][:300])
    print("Sources:", response_json["sources"])
else:
    print(f"\nError: Received status code {r1.status_code}")
    print("Error details:", r1.json())

Backend URL: https://response-adsl-remind-partly.trycloudflare.com
INFO:     35.197.82.190:0 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'chunks': 944}
INFO:     35.197.82.190:0 - "GET /papers HTTP/1.1" 200 OK
[{'arxiv_id': '1706.03762', 'title': 'Attention Is All You Need', 'authors': ['Ashish Vaswani', 'Noam Shazeer', 'Niki Parmar']}, {'arxiv_id': '1810.04805', 'title': 'BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding', 'authors': ['Jacob Devlin', 'Ming-Wei Chang', 'Kenton Lee']}, {'arxiv_id': '2004.12832', 'title': 'ColBERT: Efficient and Effective Passage Search via Contextualized Late Interaction over BERT', 'authors': ['Omar Khattab', 'Matei Zaharia']}, {'arxiv_id': '1908.10084', 'title': 'Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks', 'authors': ['Nils Reimers', 'Iryna Gurevych']}, {'arxiv_id': '2004.04906', 'title': 'Dense Passage Retrieval for Open-Domain Question Answering', 'authors': ['Vladimir Karpukhin', 'Barlas Oğuz